# 5-mer Context Mutation Rate Pipeline

Calculates context-specific mutation rates for a genomic region of interest (e.g. TSS)
using rare variant SNPs and a random intergenic background.

**Pipeline overview:**
1. **Intergenic background** — compute base mutation rates and per-5-mer context rates from intergenic SNPs and reference sequence
2. **Region-specific rate** — apply background context rates to SNPs mapped to the region of interest, producing a rate per position

**Output:** `{REGION_NAME}_5mer_rate.txt` — columns: Position, Change (weighted sum), Count (raw sum), Rate (Change/Count)
The Rate column can be multiplied by the intergenic base mutation rate to obtain an absolute mutation rate per position.

## Configuration
Set all file paths and parameters here before running.

In [ ]:
# ── Intergenic background ─────────────────────────────────────────────────
INTERGENIC_FASTA    = "/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/human_intergenic_random_1kb.fa"
INTERGENIC_SNP_FILE = "/home/alexpalazzo1/Documents/Tina/Rare_SNP/rareSNP_mapped_random_intergenic_rare.txt"
REFERENCE_FASTA     = "/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa"

# ── Region of interest ────────────────────────────────────────────────────
REGION_SNP_FILE = "/home/alexpalazzo1/Documents/Tina/Rare_SNP/rareSNP_mapped_TSS_rare.txt"
REGION_NAME     = "TSS"   # used for output file naming

# ── Base mutation rates ───────────────────────────────────────────────────
# Odds-transformed rates (p / 1-p) calculated in Section 1.1 below.
# Update these values after running Section 1.1.
BASE_MUTATION_RATES = {
    'A': 0.133144 / (1 - 0.133144),
    'T': 0.131665 / (1 - 0.131665),
    'C': 0.165358 / (1 - 0.165358),
    'G': 0.164768 / (1 - 0.164768),
}

---
# Part 1: Intergenic Background

## 1.1 Base mutation rate

Counts non-CpG bases in the intergenic FASTA (excluding C followed by G, and G preceded by C)
and counts how many of each base carry a rare variant.
Rate = mutations / total bases, expressed as an odds-transformed rate p/(1−p).

**→ Update `BASE_MUTATION_RATES` in the config cell with the values printed below.**

In [ ]:
def count_bases_in_fasta(file_path):
    """Count non-CpG A/T/C/G bases in a FASTA file."""
    counts = {'A': 0, 'T': 0, 'C': 0, 'G': 0}
    with open(file_path, 'r') as file:
        sequence = ""
        for line in file:
            if line.startswith('>'):
                if sequence:
                    _count_non_cpg_bases(sequence, counts)
                    sequence = ""
            else:
                sequence += line.strip().upper()
        if sequence:
            _count_non_cpg_bases(sequence, counts)
    return counts

def _count_non_cpg_bases(sequence, counts):
    """Count bases, excluding C in CpG (C followed by G) and G in CpG (G preceded by C)."""
    for i, base in enumerate(sequence):
        if base not in counts:
            continue
        if base == 'G' and i > 0 and sequence[i-1] == 'C':
            continue
        if base == 'C' and i < len(sequence) - 1 and sequence[i+1] == 'G':
            continue
        counts[base] += 1

def count_mutations_from_file(file_path):
    """Count how many times each base mutates (from column 2, format 'A>G')."""
    mutation_counts = {'A': 0, 'T': 0, 'C': 0, 'G': 0}
    with open(file_path, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) >= 2:
                mutation = parts[1]
                if '>' in mutation and len(mutation) == 3:
                    ref_base = mutation[0]
                    if ref_base in mutation_counts:
                        mutation_counts[ref_base] += 1
    return mutation_counts

# ── Run ───────────────────────────────────────────────────────────────────
print("Reading intergenic FASTA...")
base_counts = count_bases_in_fasta(INTERGENIC_FASTA)

print("Reading intergenic SNP file...")
mutation_counts_bg = count_mutations_from_file(INTERGENIC_SNP_FILE)

print()
print(f"{'Base':<6} {'Total Bases':>15} {'Mutations':>12} {'Rate':>12} {'Odds Rate':>12}")
print("-" * 60)
for base in ['A', 'T', 'C', 'G']:
    n_bases = base_counts[base]
    n_mut   = mutation_counts_bg[base]
    rate    = n_mut / n_bases if n_bases > 0 else 0
    odds    = rate / (1 - rate) if rate < 1 else float('inf')
    print(f"{base:<6} {n_bases:>15,} {n_mut:>12,} {rate:>12.6f} {odds:>12.6f}")
print()
print("→ Copy the Odds Rate values into BASE_MUTATION_RATES in the config cell.")

## 1.2 Background 5-mer context counts

Scans the intergenic FASTA and counts every 5-mer context genome-wide,
grouped by the centre base. This provides the background frequency of each
sequence context in intergenic regions.

**Output:** `5mer_counts_detailed.txt`

In [ ]:
from collections import defaultdict
import csv

def count_5mer_contexts(fasta_file):
    """Count all 5-mer contexts in a FASTA file, grouped by centre base."""
    counts = {base: defaultdict(int) for base in ['A', 'T', 'C', 'G']}
    with open(fasta_file, 'r') as file:
        sequences = []
        current_sequence = []
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                if current_sequence:
                    sequences.append(''.join(current_sequence))
                    current_sequence = []
            else:
                current_sequence.append(line.upper())
        if current_sequence:
            sequences.append(''.join(current_sequence))
    for seq in sequences:
        for i in range(len(seq) - 4):
            kmer = seq[i:i+5]
            if all(base in 'ATCG' for base in kmer):
                counts[kmer[2]][kmer] += 1
    return counts

def write_5mer_counts(counts, output_file):
    """Write 5-mer context counts to file, grouped by centre base."""
    with open(output_file, 'w') as f:
        for center_base in ['A', 'T', 'C', 'G']:
            contexts = counts[center_base]
            total = sum(contexts.values())
            f.write(f"CENTER BASE: {center_base}\n")
            f.write(f"Total occurrences: {total:,}\n")
            f.write(f"Unique contexts: {len(contexts):,}\n")
            f.write("-" * 60 + "\n")
            f.write(f"{'5-mer':<10} {'Count':>15} {'Percentage':>15}\n")
            f.write("-" * 60 + "\n")
            for kmer, count in sorted(contexts.items(), key=lambda x: x[1], reverse=True):
                f.write(f"{kmer:<10} {count:>15,} {(count/total)*100:>14.6f}%\n")
            f.write("\n" + "=" * 80 + "\n\n")

# ── Run ───────────────────────────────────────────────────────────────────
print("Counting 5-mer contexts in intergenic FASTA...")
context_counts = count_5mer_contexts(INTERGENIC_FASTA)
write_5mer_counts(context_counts, "5mer_counts_detailed.txt")
print("✓ Saved to 5mer_counts_detailed.txt")

## 1.3 Extract 5-mer sequences at intergenic mutation sites

**Step 1** — Generate BED coordinates spanning ±2 bases around each SNP position.  
**Step 2** — Use BEDTools to extract the reference 5-mer sequence at each site.  
**Step 3** — Merge the extracted sequences back with the SNP file.

**Outputs:**
- `rareSNP_mapped_random_intergenic_rare_5mer_coord.txt`
- `rareSNP_mapped_random_intergenic_rare_5mer_around.fa`
- `rareSNP_mapped_random_intergenic_rare_5mer_around_nucleotide.txt`

In [ ]:
import csv

def make_5mer_bed_coords(input_file, output_file):
    """
    For each SNP, generate BED coordinates spanning ±2 bases (5-mer window).
    Reads chromosome from second-to-last column, position from last column.
    """
    with open(input_file, mode='r') as infile, open(output_file, mode='w', newline='') as outfile:
        reader = csv.reader(infile, delimiter=' ')
        writer = csv.writer(outfile, delimiter='\t')
        for row in reader:
            if len(row) < 2:
                continue
            chrom = row[-2]
            pos   = float(row[-1])
            writer.writerow([chrom, int(pos - 3), int(pos + 2)])

# Step 1: Generate BED coordinates
make_5mer_bed_coords(INTERGENIC_SNP_FILE,
                     "rareSNP_mapped_random_intergenic_rare_5mer_coord.txt")
print("✓ BED coordinates saved")

In [ ]:
%%bash
# Step 2: Extract 5-mer sequences from reference genome using BEDTools
bedtools getfasta \
    -fi /media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa \
    -bed rareSNP_mapped_random_intergenic_rare_5mer_coord.txt \
    -fo rareSNP_mapped_random_intergenic_rare_5mer_around.fa \
    -s -name
echo "✓ 5-mer FASTA sequences extracted"

In [ ]:
def merge_snp_with_fasta(snp_file, fasta_file, output_file):
    """Append the extracted 5-mer sequence to each line of the SNP file."""
    with open(snp_file, 'r') as f1:
        lines1 = f1.readlines()
    with open(fasta_file, 'r') as f2:
        lines2 = [l for l in f2 if not l.startswith(">")]
    if len(lines1) != len(lines2):
        raise ValueError(f"Line count mismatch: SNP file has {len(lines1)}, FASTA has {len(lines2)}")
    with open(output_file, 'w') as out:
        for l1, l2 in zip(lines1, lines2):
            out.write(l1.strip() + "\t" + l2)

# Step 3: Merge SNP file with extracted sequences
merge_snp_with_fasta(
    INTERGENIC_SNP_FILE,
    "rareSNP_mapped_random_intergenic_rare_5mer_around.fa",
    "rareSNP_mapped_random_intergenic_rare_5mer_around_nucleotide.txt"
)
print("✓ Merged file saved to rareSNP_mapped_random_intergenic_rare_5mer_around_nucleotide.txt")

## 1.4 Count 5-mer contexts at intergenic mutation sites

Tallies how many times each 5-mer context appears at an actual mutation site,
grouped by reference base. The middle base of the 5-mer is the mutated base.

**Output:** `mutation_5mer_counts_detailed.txt`

In [ ]:
from collections import defaultdict

def count_mutation_5mer_contexts(mutation_file):
    """
    Count mutations grouped by their 5-mer context and reference base.
    Returns:
        by_reference: {base: {5mer: count}}
    """
    bases = ['A', 'T', 'C', 'G']
    by_reference = {base: defaultdict(int) for base in bases}

    with open(mutation_file, 'r') as file:
        for line_num, line in enumerate(file, 1):
            parts = line.strip().split()
            if len(parts) < 14:
                continue
            mutation_type = parts[1]
            five_mer      = parts[-1].upper()
            if len(five_mer) != 5 or not all(b in 'ATCG' for b in five_mer):
                continue
            ref_base = mutation_type[0]
            if ref_base in by_reference:
                by_reference[ref_base][five_mer] += 1

    return by_reference

def write_mutation_5mer_counts(by_reference, output_file):
    """Write mutation 5-mer counts to file, grouped by reference base."""
    with open(output_file, 'w') as f:
        for ref_base in ['A', 'T', 'C', 'G']:
            contexts = by_reference[ref_base]
            total = sum(contexts.values())
            if total == 0:
                continue
            f.write(f"REFERENCE BASE: {ref_base}\n")
            f.write(f"Total mutations: {total:,}\n")
            f.write(f"Unique contexts: {len(contexts):,}\n")
            f.write("-" * 60 + "\n")
            f.write(f"{'5-mer':<10} {'Count':>15} {'Percentage':>15}\n")
            f.write("-" * 60 + "\n")
            for kmer, count in sorted(contexts.items(), key=lambda x: x[1], reverse=True):
                f.write(f"{kmer:<10} {count:>15,} {(count/total)*100:>14.6f}%\n")
            f.write("\n" + "=" * 80 + "\n\n")

# ── Run ───────────────────────────────────────────────────────────────────
print("Counting mutation 5-mer contexts...")
mutation_5mer_counts = count_mutation_5mer_contexts(
    "rareSNP_mapped_random_intergenic_rare_5mer_around_nucleotide.txt")
write_mutation_5mer_counts(mutation_5mer_counts, "mutation_5mer_counts_detailed.txt")
print("✓ Saved to mutation_5mer_counts_detailed.txt")

## 1.5 Build master background file

For each non-CpG 5-mer, computes:
- `raw_rate` = mutation_count / background_count
- `odds_rate` = raw_rate / (1 − raw_rate)  *(odds transformation)*
- `change` = odds_rate / base_odds_rate  *(enrichment relative to per-nucleotide baseline)*

**Outputs:** `master_5mer_background.csv`, `master_5mer_background_nonCpG.csv`

In [ ]:
import csv
from collections import defaultdict

def load_mutation_counts(mutation_file):
    """Load mutation 5-mer counts from mutation_5mer_counts_detailed.txt."""
    mutation_counts = {base: defaultdict(int) for base in ['A', 'T', 'C', 'G']}
    current_base = None
    with open(mutation_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("REFERENCE BASE:"):
                current_base = line.split(":")[1].strip()
            elif current_base and line and not line.startswith(('-', '=')):
                parts = line.split()
                if len(parts) >= 2 and len(parts[0]) == 5 and all(c in 'ATCG' for c in parts[0]):
                    try:
                        mutation_counts[current_base][parts[0]] = int(parts[1].replace(',', ''))
                    except ValueError:
                        pass
    return mutation_counts

def load_background_counts(background_file):
    """Load background 5-mer counts from 5mer_counts_detailed.txt."""
    background_counts = {base: defaultdict(int) for base in ['A', 'T', 'C', 'G']}
    current_base = None
    with open(background_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("CENTER BASE:") or line.startswith("REFERENCE BASE:"):
                current_base = line.split(":")[1].strip()
            elif current_base and line and not line.startswith(('-', '=')):
                parts = line.split()
                if len(parts) >= 2 and len(parts[0]) == 5 and all(c in 'ATCG' for c in parts[0]):
                    try:
                        background_counts[current_base][parts[0]] = int(parts[1].replace(',', ''))
                    except ValueError:
                        pass
    return background_counts

def create_master_file_csv(mutation_counts, background_counts, base_mutation_rates, output_file):
    """
    Create master CSV with columns:
    Context, Reference_Base, Mutation_Count, Background_Count,
    Raw_Rate, Odds_Rate, Base_Mutation_Rate_Odds, Change
    """
    with open(output_file, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Context', 'Reference_Base', 'Mutation_Count', 'Background_Count',
                         'Raw_Rate', 'Odds_Rate', 'Base_Mutation_Rate_Odds', 'Change'])
        for base in ['A', 'T', 'C', 'G']:
            mut_contexts = mutation_counts.get(base, {})
            bg_contexts  = background_counts.get(base, {})
            base_rate    = base_mutation_rates[base]
            all_kmers    = set(mut_contexts.keys()) | set(bg_contexts.keys())
            for kmer in sorted(all_kmers):
                mut_count = mut_contexts.get(kmer, 0)
                bg_count  = bg_contexts.get(kmer, 0)
                if bg_count == 0:
                    continue
                raw_rate = mut_count / bg_count
                if raw_rate >= 1:
                    continue   # skip edge cases
                odds_rate = raw_rate / (1 - raw_rate)
                change    = odds_rate / base_rate
                writer.writerow([kmer, base, mut_count, bg_count,
                                 f"{raw_rate:.8f}", f"{odds_rate:.8f}",
                                 f"{base_rate:.6f}", f"{change:.8f}"])

def is_cpg_context(context):
    """Return True if the centre base of the 5-mer is part of a CpG site."""
    mid = context[2]
    if mid == 'C' and context[3] == 'G':
        return True
    if mid == 'G' and context[1] == 'C':
        return True
    return False

def filter_cpg_from_csv(input_file, output_file):
    """Remove rows where the context is a CpG site."""
    with open(input_file, 'r') as infile, open(output_file, 'w', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)
        writer.writerow(next(reader))   # header
        kept = filtered = 0
        for row in reader:
            if not is_cpg_context(row[0]):
                writer.writerow(row)
                kept += 1
            else:
                filtered += 1
    print(f"  Kept: {kept:,}  |  Filtered (CpG): {filtered:,}")

# ── Run ───────────────────────────────────────────────────────────────────
print("Loading counts...")
mutation_counts  = load_mutation_counts("mutation_5mer_counts_detailed.txt")
background_counts = load_background_counts("5mer_counts_detailed.txt")

print("Building master background file...")
create_master_file_csv(mutation_counts, background_counts,
                       BASE_MUTATION_RATES, "master_5mer_background.csv")
print("✓ Saved to master_5mer_background.csv")

print("Filtering CpG contexts...")
filter_cpg_from_csv("master_5mer_background.csv", "master_5mer_background_nonCpG.csv")
print("✓ Saved to master_5mer_background_nonCpG.csv")

---
# Part 2: Region-Specific Context Rate

## 2.1 Extract 5-mer sequences at region mutation sites

Same three steps as Section 1.3, applied to the region of interest (e.g. TSS).

**Outputs:**
- `rareSNP_mapped_{REGION_NAME}_5mer_coord.txt`
- `rareSNP_mapped_{REGION_NAME}_5mer_around.fa`
- `rareSNP_mapped_{REGION_NAME}_5mer_around_nucleotide.txt`

In [ ]:
import csv

# Step 1: Generate BED coordinates
make_5mer_bed_coords(REGION_SNP_FILE,
                     f"rareSNP_mapped_{REGION_NAME}_5mer_coord.txt")
print(f"✓ BED coordinates saved to rareSNP_mapped_{REGION_NAME}_5mer_coord.txt")

In [ ]:
%%bash
# Step 2: Extract 5-mer sequences from reference genome using BEDTools
bedtools getfasta \
    -fi /media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa \
    -bed rareSNP_mapped_${REGION_NAME}_5mer_coord.txt \
    -fo rareSNP_mapped_${REGION_NAME}_5mer_around.fa \
    -s -name
echo "✓ 5-mer FASTA sequences extracted"

In [ ]:
# Step 3: Merge SNP file with extracted sequences
merge_snp_with_fasta(
    REGION_SNP_FILE,
    f"rareSNP_mapped_{REGION_NAME}_5mer_around.fa",
    f"rareSNP_mapped_{REGION_NAME}_5mer_around_nucleotide.txt"
)
print(f"✓ Merged file saved to rareSNP_mapped_{REGION_NAME}_5mer_around_nucleotide.txt")

## 2.2 Build position × 5-mer count matrix

Reads the merged mutation file and counts, for each 5-mer context, how many
mutations occur at each position relative to the feature anchor.
Produces a 1024 × 1001 matrix (all possible 5-mers × positions 0–1000).

**Output:** `{REGION_NAME}_5mer_master_file_positions_0_to_1000.txt`

In [ ]:
import pandas as pd
from itertools import product

def create_5mer_position_matrix(input_file, output_file, max_position=1000):
    """
    Create a (1024 5-mers) × (positions 0–max_position) count matrix.
    Input file columns: position (col 1), ..., 5-mer context (last col).
    """
    nucleotides = ['A', 'C', 'G', 'T']
    all_5mers   = [''.join(c) for c in product(nucleotides, repeat=5)]
    counts      = {pos: {kmer: 0 for kmer in all_5mers}
                   for pos in range(max_position + 1)}
    n_processed = 0

    with open(input_file, 'r') as f:
        for line_num, line in enumerate(f, 1):
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            try:
                position = int(parts[0])
                context  = parts[-1]
                if 0 <= position <= max_position and context in counts[position]:
                    counts[position][context] += 1
                    n_processed += 1
            except ValueError:
                print(f"Warning: Could not parse line {line_num}: {line.strip()}")

    data    = [[kmer] + [counts[pos][kmer] for pos in range(max_position + 1)]
               for kmer in sorted(all_5mers)]
    columns = ['5-mer_context'] + [str(pos) for pos in range(max_position + 1)]
    df = pd.DataFrame(data, columns=columns)
    df.to_csv(output_file, sep='\t', index=False)
    print(f"  Lines processed: {n_processed:,}")
    print(f"  Positions with data: {sum(1 for pos in range(max_position+1) if any(counts[pos].values()))}")
    return df

# ── Run ───────────────────────────────────────────────────────────────────
output_matrix = f"{REGION_NAME}_5mer_master_file_positions_0_to_1000.txt"
print(f"Building position × 5-mer matrix...")
df_matrix = create_5mer_position_matrix(
    f"rareSNP_mapped_{REGION_NAME}_5mer_around_nucleotide.txt",
    output_matrix)
print(f"✓ Saved to {output_matrix}")

## 2.3 Filter CpG contexts from position matrix

Removes rows where the centre base of the 5-mer is part of a CpG site.

**Output:** `{REGION_NAME}_5mer_master_file_positions_0_to_1000_nonCpG.txt`

In [ ]:
import pandas as pd

def filter_cpg_from_matrix(input_file, output_file):
    """Remove 5-mer rows that are CpG contexts from the position matrix."""
    df = pd.read_csv(input_file, sep='\t')
    mask = df['5-mer_context'].apply(lambda mer: not is_cpg_context(mer))
    filtered_df = df[mask]
    filtered_df.to_csv(output_file, sep='\t', index=False)
    print(f"  Kept {mask.sum():,} / {len(df):,} contexts "
          f"({(~mask).sum():,} CpG contexts removed)")
    return filtered_df

# ── Run ───────────────────────────────────────────────────────────────────
output_nonCpG = f"{REGION_NAME}_5mer_master_file_positions_0_to_1000_nonCpG.txt"
print("Filtering CpG contexts from position matrix...")
df_nonCpG = filter_cpg_from_matrix(
    f"{REGION_NAME}_5mer_master_file_positions_0_to_1000.txt",
    output_nonCpG)
print(f"✓ Saved to {output_nonCpG}")

## 2.4 Apply change values

Multiplies each row of the non-CpG position matrix by the corresponding
`change` value from `master_5mer_background_nonCpG.csv`.
This weights each count by the relative mutability of its 5-mer context.

**Output:** `{REGION_NAME}_5mer_change.txt`

In [ ]:
import csv

def apply_change_values(matrix_file, background_csv, output_file):
    """
    Multiply each row of the position matrix by its context-specific change value.
    Rows whose context is not found in the background file are kept unchanged.
    """
    # Load change values from background CSV
    context_to_change = {}
    with open(background_csv, 'r') as f:
        reader = csv.reader(f)
        next(reader)   # skip header
        for row in reader:
            if row:
                context_to_change[row[0]] = float(row[-1])  # Context → Change

    with open(matrix_file, 'r') as f_in, open(output_file, 'w') as f_out:
        header = f_in.readline().strip()
        f_out.write(header + '\n')
        missing = 0
        for line in f_in:
            parts   = line.strip().split('\t')
            context = parts[0]
            if context in context_to_change:
                change    = context_to_change[context]
                new_parts = [context]
                for val in parts[1:]:
                    try:
                        new_parts.append(f"{int(val) * change:.2f}")
                    except ValueError:
                        new_parts.append(val)
                f_out.write('\t'.join(new_parts) + '\n')
            else:
                missing += 1
                f_out.write(line.strip() + '\n')
    if missing:
        print(f"  Warning: {missing} contexts not found in background file (kept unchanged)")

# ── Run ───────────────────────────────────────────────────────────────────
output_change = f"{REGION_NAME}_5mer_change.txt"
print("Applying change values...")
apply_change_values(
    output_nonCpG,
    "master_5mer_background_nonCpG.csv",
    output_change)
print(f"✓ Saved to {output_change}")

## 2.5 Calculate rate per position

For each position:

$$\text{Rate} = \frac{\sum_{kmer} \text{count}_{kmer} \times \text{change}_{kmer}}{\sum_{kmer} \text{count}_{kmer}}$$

This is the mutation-weighted average enrichment relative to the intergenic base rate.
Multiply the Rate column by the intergenic base mutation rate to obtain an absolute rate.

**Output:** `{REGION_NAME}_5mer_rate.txt`

In [ ]:
import pandas as pd
import numpy as np

def calculate_rate_per_position(change_file, count_file, output_file):
    """
    Calculate Rate = sum(weighted counts) / sum(raw counts) per position.
    file1 = change-weighted matrix  (from 2.4)
    file2 = raw count matrix        (from 2.3)
    """
    df_change = pd.read_csv(change_file, sep='\t')
    df_counts  = pd.read_csv(count_file,  sep='\t')

    weighted_sums = df_change.iloc[:, 1:].sum(axis=0).values
    raw_sums      = df_counts.iloc[:,  1:].sum(axis=0).values

    with np.errstate(divide='ignore', invalid='ignore'):
        rate = np.where(raw_sums != 0, weighted_sums / raw_sums, np.nan)

    results = pd.DataFrame({
        'Position': range(1, len(rate) + 1),
        'Change':   weighted_sums,
        'Count':    raw_sums,
        'Rate':     rate
    })
    results.to_csv(output_file, sep='\t', index=False, float_format='%.6f')
    print(f"  Mean rate: {np.nanmean(rate):.6f}")
    print(f"  Rate range: {np.nanmin(rate):.6f} – {np.nanmax(rate):.6f}")

# ── Run ───────────────────────────────────────────────────────────────────
output_rate = f"{REGION_NAME}_5mer_rate.txt"
print("Calculating rate per position...")
calculate_rate_per_position(output_change, output_nonCpG, output_rate)
print(f"✓ Saved to {output_rate}")

## 2.6 Plot

Rate vs position relative to the feature anchor.
The dashed grey line at y=1 represents the intergenic baseline.

**Output:** `{REGION_NAME}_5mer_rate_plot.png`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_rate(input_file, output_image=None, region_name="Region"):
    df = pd.read_csv(input_file, sep='\t')

    plt.figure(figsize=(12, 5))
    plt.plot(df['Position'], df['Rate'], 'b-', linewidth=1, alpha=0.8, label='Rate')
    plt.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='Baseline (y=1)')
    plt.xlabel('Position', fontsize=12)
    plt.ylabel('Rate', fontsize=12)
    plt.title(f'5-mer Context Mutation Rate — {region_name}', fontsize=13)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    if output_image:
        plt.savefig(output_image, dpi=300, bbox_inches='tight')
        print(f"✓ Plot saved to {output_image}")
    plt.show()

# ── Run ───────────────────────────────────────────────────────────────────
plot_rate(output_rate,
          output_image=f"{REGION_NAME}_5mer_rate_plot.png",
          region_name=REGION_NAME)